# nb_05d — Reflection-equivariant encoder: build-in vs augment (E3)

E2 imposed the group action on the *read-out*. **E3** lifts it to the **encoder**, two ways:

- **E3a — frame-averaged encoder (exact, zero-retrain).** Wrap the frozen encoder so the *whole latent field* transforms correctly under reflection.
- **E3b — reflection augmentation (does it emerge?).** Retrain Stage-0 with a mirror augmentation and ask whether equivariance is *induced* (Benton et al., 2020).

> **Responsible-use notice.** All results in this notebook are **transductive** — the
> S-JEPA encoder was trained on the very sequences being evaluated — so they carry
> *internal validity only* and make **no** claim of generalization to new sources or
> people. The **source video** (not the clip, not the individual) is the independent unit
> of analysis. The dataset's condition folders (`normal`, `parkinsons`, `stroke`,
> `myopathic`, `cerebralpalsy`) are **dataset annotations, not diagnoses**. The dataset's
> official distribution provides annotations and public video URLs, not raw video; this
> analysis uses derived pose sequences, infers no identity, and redistributes no raw or
> identity-bearing frames. **No institutional ethics determination or completed data-use
> review is yet on record; both must be resolved before submission.**

## E3a. Frame-averaged encoder $E'(x)=\tfrac12\big(E(x)+\sigma\!\cdot\!E(Mx)\big)$

With $\sigma$ the left/right **token** permutation, the wrapped encoder is **exactly token-level reflection-equivariant**, $E'(Mx)=\sigma\cdot E'(x)$, to machine precision and with **no retraining**. Consequently the laterality feature on $E'$ splits *exactly* into an antisymmetric block (the $\ell-r$ channels; slope $-1$ for any nonzero read-out) and a symmetric block (the $\ell+r$ channels). The antisymmetric block **corresponds to** E2's $\Phi$: it is one half of $\Phi$'s $\ell-r$ sub-block (up to standardization), while $\Phi$ additionally antisymmetrizes the $\ell+r$ block — so E3a and E2 are two views of the same frame-averaging construction, not identical features.

**Honest boundary.** A *free* ridge on the *full* $E'$ feature still gives slope $\approx-0.77$: encoder equivariance alone does **not** force an antisymmetric *decoder* — the read-out must still select the antisymmetric part. That boundary is exactly what motivates E3b (try to *train* the symmetry in).

In [1]:
import json, os, subprocess, sys, textwrap
from pathlib import Path

def find_experiment_dir(start=None):
    candidates = []
    if os.getenv("ALEXPOSE_ROOT"):
        env_root = Path(os.environ["ALEXPOSE_ROOT"]).expanduser().resolve()
        candidates.extend([env_root, env_root / "experiments" / "sjepa" / "gavd5-drift"])
    start = Path(start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        candidates.extend([base, base / "experiments" / "sjepa" / "gavd5-drift"])
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "work" / "experiments").is_dir():
            return candidate
    raise FileNotFoundError(f"Cannot locate gavd5-drift from {start}; set ALEXPOSE_ROOT.")


EXPERIMENT_DIR = find_experiment_dir()
NOTEBOOK_DIR = EXPERIMENT_DIR / "neurips-brain-body"
PY = EXPERIMENT_DIR / ".venv" / "bin" / "python"
PY = str(PY if PY.exists() else sys.executable)   # fall back to the running kernel
ART = EXPERIMENT_DIR / "work" / "artifacts" / "real"

def run_experiment(script_relpath):
    """Run a validated standalone experiment script; it regenerates its JSON artifact."""
    script = EXPERIMENT_DIR / script_relpath
    print(f"running {script.name} with {PY} ...")
    proc = subprocess.run([PY, str(script)], cwd=str(EXPERIMENT_DIR),
                          capture_output=True, text=True)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print("STDERR (tail):\n", proc.stderr[-4000:])
        raise RuntimeError(f"{script.name} exited {proc.returncode}")
    return proc

def approx(a, b, tol):
    return abs(float(a) - float(b)) <= tol


In [2]:
# Re-run E3a end-to-end (writes the top-level of idea9_equivariant_encoder_result.json).
run_experiment('work/experiments/e3a_frame_averaging_wrapper.py')
res = json.loads((ART / 'idea9_equivariant_encoder_result.json').read_text())
prim = res['primary_cohort']
ex = prim['exactness']; lanes = prim['lanes']; slopes = prim['mirror_slopes']; ci = prim['repeated_cv_ci95']
print('E3a exactness (all should be 0.0 to machine precision):')
for k, v in ex.items():
    print(f'  {k:38s} {v:.2e}')
def band(k):
    d = ci[k]; return f"{d['mean']:.3f} [{d['ci95_lo']:.3f}, {d['ci95_hi']:.3f}]"
print()
print(f"  A'_diff learned (antisym block)  R2={band('Aprime_diff_learned'):24s} slope={slopes['Aprime_diff_learned']:+.4f}")
print(f"  A'_diff floor  (untrained)       R2={band('Aprime_diff_floor'):24s} slope={slopes['Aprime_diff_floor']:+.4f}")
print(f"  A' free ridge (full feature)                                    slope={slopes['Aprime_free']:+.4f}  <- NOT -1")

running e3a_frame_averaging_wrapper.py with /Users/pmui/dev/alexpose/experiments/sjepa/gavd5-drift/.venv/bin/python ...


checkpoint 7d13841aceac config={'frames': 64, 'joints': 33, 'coordinate_dim': 3, 'segment_length': 4, 'embed_dim': 96, 'encoder_depth': 4, 'predictor_depth': 2, 'heads': 4} train_ids=626
642 availability: 642 seq / 94 sources
626 modeled     : 626 seq / 93 sources

=== E3(a) PRIMARY: 626 modeled ===
{
  "exactness": {
    "token_equivariance_max_abs_err": 0.0,
    "diff_block_antisymmetry_max_abs_err": 0.0,
    "sum_block_symmetry_max_abs_err": 0.0
  },
  "lanes": {
    "Aprime_free": {
      "r2": 0.27609425541823396,
      "mae": 1.9914854114150713,
      "sign_consistency": 0.5161290322580645
    },
    "Aprime_diff_learned": {
      "r2": 0.2880836772599953,
      "mae": 1.9548007698430676,
      "sign_consistency": 0.5483870967741935
    },
    "Aprime_diff_floor": {
      "r2": 0.18687885322023934,
      "mae": 2.0572870388460225,
      "sign_consistency": 0.5698924731182796
    },
    "B_raw": {
      "r2": 0.9999999999968954,
      "mae": 3.545913241042281e-06,
      "sign_cons

In [3]:
# ---- Assert E3a exactness + that E3a's antisym block reproduces E2's Phi
assert ex['token_equivariance_max_abs_err'] < 1e-4, ex
assert ex['diff_block_antisymmetry_max_abs_err'] < 1e-4, ex
assert ex['sum_block_symmetry_max_abs_err'] < 1e-4, ex
assert approx(slopes['Aprime_diff_learned'], -1.0, 1e-3), slopes['Aprime_diff_learned']
assert approx(ci['Aprime_diff_learned']['mean'], 0.253, 0.02), ci['Aprime_diff_learned']
assert slopes['Aprime_free'] > -1.0 + 0.05, 'free ridge on full E-prime must NOT be -1'
print('OK — E3a: exact token-level reflection-equivariance, zero retrain.')

OK — E3a: exact token-level reflection-equivariance, zero retrain.


## E3b. Does reflection augmentation *induce* it? (a single-seed negative)

We retrain **Stage-0** (the `normal`-annotated rows only, 270 sequences / 29 sources, 300 epochs, single seed on MPS) with **one switch**: sample-level consistent reflection augmentation **on** ($p=0.5$) vs **off** ($p=0.0$), identical trainer/seed/hardware. `canonical` is the original Stage-0 checkpoint, a fidelity cross-check.

> **Provenance.** The ~15-minute-per-arm MPS training lives in `../work/experiments/e3b_reflection_augmented_retrain.py` and produced the checkpoints `sjepa_normal_e3b_flip0p00.pt` / `…flip0p50.pt` (final JEPA $0.720$ / $0.807$). The cell below **loads** those checkpoints and probes them — it does **not** retrain — so the notebook stays fast and deterministic while every probe number is reproduced from scratch.

**Primary metric** = the *free-readout mirror slope*: does augmentation move it toward $-1$? And does the signed axis clear the untrained floor?

In [4]:
# Probe the three Stage-0 encoders + floor (loads pre-trained checkpoints; no retraining).
run_experiment('work/experiments/e3b_probe_and_merge.py')
res = json.loads((ART / 'idea9_equivariant_encoder_result.json').read_text())
rb = res['retrain']; enc = rb['primary_cohort']['encoders']; tr = rb['training']
print('training: arm_off finalJEPA', tr['arm_off']['final_jepa'],
      '| arm_on finalJEPA', tr['arm_on']['final_jepa'])
print('\nE3b — 270 normal-annotated (transductive)   A_free R2 [stability interval]   free-readout slope')
for name in ('arm_on', 'arm_off', 'canonical', 'floor_untrained'):
    e = enc[name]; d = e['A_free_r2_ci95']
    print(f"  {name:16s} {d['mean']:+.3f} [{d['ci95_lo']:+.3f}, {d['ci95_hi']:+.3f}]   "
          f"slope={e['A_free_mirror_slope']:+.3f}")

running e3b_probe_and_merge.py with /Users/pmui/dev/alexpose/experiments/sjepa/gavd5-drift/.venv/bin/python ...


config={'frames': 64, 'joints': 33, 'coordinate_dim': 3, 'segment_length': 4, 'embed_dim': 96, 'encoder_depth': 4, 'predictor_depth': 2, 'heads': 4}
canonical flip=n/a seqs=270
arm_off   flip=0.0 seqs=270 finalJEPA=0.7203
arm_on    flip=0.5 seqs=270 finalJEPA=0.8069

=== E3(b) PRIMARY: 270 normal (transductive) ===
  canonical        A_free r2=+0.051 [-0.075,+0.031]  free_slope=-0.553  Phi r2=+0.263
  arm_off          A_free r2=+0.069 [-0.113,+0.016]  free_slope=-0.753  Phi r2=+0.174
  arm_on           A_free r2=+0.151 [+0.011,+0.114]  free_slope=-0.510  Phi r2=+0.229
  floor_untrained  A_free r2=+0.189 [+0.013,+0.144]  free_slope=-0.818  Phi r2=+0.259
  B_raw ceiling r2=1.0000

=== E3(b) ROBUSTNESS: 626 (extrapolation) ===
  canonical        A_free r2=+0.265  free_slope=-0.646  Phi r2=+0.262
  arm_off          A_free r2=+0.183  free_slope=-0.761  Phi r2=+0.141
  arm_on           A_free r2=+0.217  free_slope=-0.626  Phi r2=+0.337
  floor_untrained  A_free r2=+0.229  free_slope=-0.787  

In [5]:
# ---- Assert the paper's E3b single-seed negative: augmentation does NOT buy equivariance
on  = enc['arm_on']['A_free_r2_ci95']; floor = enc['floor_untrained']['A_free_r2_ci95']
assert approx(on['mean'], 0.062, 0.02), on
assert approx(floor['mean'], 0.078, 0.02), floor
# augmented axis does NOT clear the untrained floor
assert on['mean'] <= floor['ci95_hi'], (on, floor)
# the free-readout slope does NOT move toward -1; the UNTRAINED floor is closest
s_on = enc['arm_on']['A_free_mirror_slope']; s_floor = enc['floor_untrained']['A_free_mirror_slope']
assert s_floor < s_on, (s_on, s_floor)  # floor slope (-0.82) closer to -1 than arm_on (-0.51)
print('OK — E3b: single-seed negative. Augmentation changes the loss, not the geometry;')
print('     free-readout slope does not track equivariance (untrained floor is closest to -1).')

OK — E3b: single-seed negative. Augmentation changes the loss, not the geometry;
     free-readout slope does not track equivariance (untrained floor is closest to -1).


## Synthesis — the emergent-vs-built-in ladder

The four experiments form a $2\times2$ grid (`../neurips-laterality/docs/figures/fig6_ladder.svg`): symmetry in the *read-out* or the *encoder*, *hoped for* or *built in*.

|              | Emergent (hope) | Built-in (construct) |
|--------------|-----------------|----------------------|
| **Read-out** | E1: slope $-0.70$, learned $\approx$ floor (intervals overlap), gates fail | E2: slope $-1.0000$, $0.273>0.198$ (disjoint intervals) |
| **Encoder**  | E3b: slope not toward $-1$, axis $\le$ floor | E3a: token error $0.0$, exact split |

In the audited system (a single checkpoint; a single-seed augmentation arm), standard **and** reflection-augmented self-supervision leave bilateral symmetry un-learned; frame averaging the read-out or encoder recovers it to machine precision with zero or minimal retraining — and that constructive guarantee is general, not tied to this checkpoint. For geometry-aware world models of articulated bodies, symmetry is cheap to *impose* and — at least here — not reliably obtained by *hoping* it emerges: **build the geometry in; don't hope it emerges.**

*(E3b is a single-seed, `normal`-annotated proof-of-concept; see `../neurips-laterality/docs/paper.md` §9 for the full limitations.)*